# Real Estate Data Validation & Cleanup
Validate and standardize real estate data formats: remove unit suffixes, normalize decimal separators, and identify format issues.

## Section 1: Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

## Section 2: Load and Inspect Data

In [2]:
# Load và gộp cả 2 file Excel nguồn
excel_sources = ['data/Can ho.xlsx', 'data/Nha dat.xlsx']

frames = []
for fp in excel_sources:
    tmp = pd.read_excel(fp)
    print(f"✅ {fp}: {len(tmp)} rows, {len(tmp.columns)} cols")
    frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print(f"\n📦 Sau khi gộp: {df.shape[0]} rows, {df.shape[1]} cols")
print(f"Columns: {list(df.columns)}")
print("\nFirst 3 rows:")
display(df.head(3))

Available Excel files:
  - listings.xlsx
  - exceltemplate.xlsx
  - listings_required_fields_vi.xlsx
  - listings_full.xlsx
  - listings_required_fields.xlsx
  - Nha_dat_v2.xlsx
  - hcm_nha_dat_full_detail_131_cleaned.xlsx
  - Nha dat_clean.xlsx
  - Nha dat.xlsx
  - Can ho_clean.xlsx
  - Can ho.xlsx
  - listings_default_vi.xlsx
  - Can_ho_v2.xlsx
  - hcm_can_ho_full_detail_from_template.xlsx

Dataset shape: (489, 25)
Columns: ['#', 'Mã tin', 'Loại tin', 'Ngày đăng', 'Ngày hết hạn', 'Loại BĐS', 'Tỉnh thành', 'Quận huyện', 'Folder', 'Tiêu đề', 'Địa chỉ', 'Địa chỉ cũ', 'Link Location', 'Latitude and Longitude', 'Mô tả', 'Lịch sử giá khu', 'Khoảng giá', 'Diện tích', 'Đơn giá', 'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Pháp lý', 'Nội thất', 'Hướng ban công', 'Hướng nhà']

First 3 rows:

Dataset shape: (489, 25)
Columns: ['#', 'Mã tin', 'Loại tin', 'Ngày đăng', 'Ngày hết hạn', 'Loại BĐS', 'Tỉnh thành', 'Quận huyện', 'Folder', 'Tiêu đề', 'Địa chỉ', 'Địa chỉ cũ', 'Link Location', 'Latitude and

,#,Mã tin,Loại tin,Ngày đăng,Ngày hết hạn,Loại BĐS,Tỉnh thành,Quận huyện,Folder,Tiêu đề,...,Lịch sử giá khu,Khoảng giá,Diện tích,Đơn giá,Số phòng ngủ,"Số phòng tắm, vệ sinh",Pháp lý,Nội thất,Hướng ban công,Hướng nhà
0,1,45222670,Tin VIP Kim Cương,03/03/2026,11/03/2026,Căn hộ chung cư,Hồ Chí Minh,Phường Long Bình,Bán/Hồ Chí Minh/Long Bình/Căn hộ chung cư tại ...,"Còn sót căn góc 2PN 69m2-Beverly Solari, giá t...",...,"13,2% Giá tại dự án này đã tăng trong vòng 1 n...","3,95 tỷ",69 m²,"57,25 tr/m²",2.0,2.0,Sổ đỏ/ Sổ hồng,Đầy đủ.,Tây - Nam,Đông - Bắc
1,2,41003332,Tin VIP Kim Cương,25/02/2026,12/03/2026,Căn hộ chung cư,Hồ Chí Minh,Phường Dĩ An,Bán/Hồ Chí Minh/Dĩ An/Căn hộ chung cư tại Phú ...,Lì Xì Lộc Phát lên đến 680tr cho căn 3PN - 2WC...,...,"36,5% Giá tại dự án này đã tăng trong vòng 1 n...","4,28 tỷ",104 m²,"41,12 tr/m²",3.0,2.0,Sổ đỏ/ Sổ hồng,Cơ bản,NaN,Tây - Bắc
2,3,44850086,Tin VIP Vàng,09/03/2026,16/03/2026,Căn hộ chung cư,Hồ Chí Minh,Phường An Khánh,Bán/Hồ Chí Minh/An Khánh/Căn hộ chung cư tại E...,Empire City - bán căn 1PN - đúng gu người độc ...,...,"59,1% Giá tại dự án này đã tăng trong vòng 1 n...","14,2 tỷ",64 m²,"221,88 tr/m²",1.0,1.0,HĐMB,Đầy đủ,Đông - Nam,Đông - Nam


## Section 3: Validate Price Format

In [15]:
# Check price format - should contain ₫ or be numeric
# Identify records with price issues (missing ₫, malformed)
print("=== PRICE FORMAT VALIDATION ===\n")

price_col = 'Khoảng giá'
if price_col in df.columns:
    # Check which prices have ₫ symbol
    has_ty = df[price_col].astype(str).str.contains('tỷ', na=False)
    has_dong = df[price_col].astype(str).str.contains('₫', na=False)
    
    print(f"Total rows: {len(df)}")
    print(f"Prices with 'tỷ': {has_ty.sum()}")
    print(f"Prices with '₫': {has_dong.sum()}")
    
    # Find problematic prices
    problematic = df[~(has_ty | has_dong | df[price_col].isna())]
    
    if len(problematic) > 0:
        print(f"\n❌ PROBLEMATIC PRICES (not tỷ or ₫): {len(problematic)} records")
        print("\nSample problematic records (Mã tin, Giá thành):")
        sample = problematic[['Mã tin', price_col]].head(20)
        for idx, row in sample.iterrows():
            print(f"  {row['Mã tin']}: {row[price_col]}")
    else:
        print("\n✅ All prices have tỷ or ₫ format")
else:
    print(f"Column '{price_col}' not found. Available columns: {list(df.columns)}")

=== PRICE FORMAT VALIDATION ===

Total rows: 489
Prices with 'tỷ': 488
Prices with '₫': 0

❌ PROBLEMATIC PRICES (not tỷ or ₫): 1 records

Sample problematic records (Mã tin, Giá thành):
  45217453: 99 triệu/m²


## Section 4: Remove Unit Suffixes

In [3]:
# Define cleaning functions
def clean_price(val):
    """Remove ₫ and tỷ from price, return numeric string"""
    if pd.isna(val) or val == '':
        return val
    val = str(val).strip()
    val = val.replace('₫', '').replace('tỷ', '').strip()
    return val

def clean_area(val):
    """Remove m² from area, return numeric string"""
    if pd.isna(val) or val == '':
        return val
    val = str(val).strip()
    val = val.replace('m²', '').replace('m2', '').strip()
    return val

def clean_floor(val):
    """Remove tầng from floor count, return numeric string"""
    if pd.isna(val) or val == '':
        return val
    val = str(val).strip()
    val = val.replace('tầng', '').strip()
    return val

def clean_length(val):
    """Remove m from length (mặt tiền, đường vào), return numeric string"""
    if pd.isna(val) or val == '':
        return val
    val = str(val).strip()
    val = re.sub(r'\s*m\s*$', '', val).strip()  # Remove 'm' at end
    return val

# Create a copy for cleaning
df_clean = df.copy()

# Apply cleaning
print("=== REMOVING UNIT SUFFIXES ===\n")

# Clean Khoảng giá (Price)
if 'Khoảng giá' in df_clean.columns:
    df_clean['Khoảng giá'] = df_clean['Khoảng giá'].apply(clean_price)
    print("✅ Cleaned Khoảng giá (removed ₫, tỷ)")

# Clean Diện tích (Area)
if 'Diện tích' in df_clean.columns:
    df_clean['Diện tích'] = df_clean['Diện tích'].apply(clean_area)
    print("✅ Cleaned Diện tích (removed m²)")

# Clean Số tầng (Floor count)
if 'Số tầng' in df_clean.columns:
    df_clean['Số tầng'] = df_clean['Số tầng'].apply(clean_floor)
    print("✅ Cleaned Số tầng (removed tầng)")

# Clean Mặt tiền (Frontage)
if 'Mặt tiền' in df_clean.columns:
    df_clean['Mặt tiền'] = df_clean['Mặt tiền'].apply(clean_length)
    print("✅ Cleaned Mặt tiền (removed m)")

# Clean Đường vào nhà (Street entrance)
if 'Đường vào' in df_clean.columns:
    df_clean['Đường vào'] = df_clean['Đường vào'].apply(clean_length)
    print("✅ Cleaned Đường vào nhà (removed m)")

print("\nSample of cleaned data:")
#display(df_clean[['Mã tin', 'Khoảng giá', 'Diện tích', 'Số tầng', 'Mặt tiền', 'Đường vào']].head(5))
display(df_clean[['Mã tin', 'Khoảng giá', 'Diện tích']].head(5))

=== REMOVING UNIT SUFFIXES ===

✅ Cleaned Khoảng giá (removed ₫, tỷ)
✅ Cleaned Diện tích (removed m²)

Sample of cleaned data:


,Mã tin,Khoảng giá,Diện tích
0,45222670,"3,95",69
1,41003332,"4,28",104
2,44850086,"14,2",64
3,45225431,"32,1",87
4,43489295,"7,05",181


## Section 5: Standardize Decimal Separators

In [4]:
def normalize_decimal(val):
    """Convert decimal separator to dot (.), handle Vietnamese format (comma)"""
    if pd.isna(val) or val == '':
        return val
    val = str(val).strip()
    # Replace comma with dot for decimal separator
    # Be careful: only replace commas that look like decimal separators (followed by 1-2 digits)
    val = re.sub(r',(\d{1,2})$', r'.\1', val)
    # Also handle case where there's comma followed by more digits
    val = val.replace(',', '.')
    return val

# Apply decimal normalization to numeric columns
numeric_cols = ['Khoảng giá', 'Diện tích', 'Số tầng', 'Mặt tiền', 'Đường vào']

print("=== STANDARDIZING DECIMAL SEPARATORS ===\n")

for col in numeric_cols:
    if col in df_clean.columns:
        before_count = df_clean[col].astype(str).str.contains(',', na=False).sum()
        df_clean[col] = df_clean[col].apply(normalize_decimal)
        after_count = df_clean[col].astype(str).str.contains(',', na=False).sum()
        print(f"{col}: {before_count} → {after_count} with commas")

print("\n✅ All decimal separators standardized to dot (.)")

print("\nSample of normalized data:")
display(df_clean[['Mã tin', 'Khoảng giá', 'Diện tích']].head(10))

=== STANDARDIZING DECIMAL SEPARATORS ===

Khoảng giá: 435 → 0 with commas
Diện tích: 142 → 0 with commas

✅ All decimal separators standardized to dot (.)

Sample of normalized data:


,Mã tin,Khoảng giá,Diện tích
0,45222670,3.95,69
1,41003332,4.28,104
2,44850086,14.2,64
3,45225431,32.1,87
4,43489295,7.05,181
5,44335474,6.2,67
6,45260467,10.6,87
7,45197258,3.2,44
8,45059973,6.6,104
9,45229051,2.38,55


## Section 6: Export Cleaned Data

In [5]:
# Helper: convert Excel file to CSV and JSON
# - Bỏ cột '#' nếu có
# - Lấy image_list từ field 'Danh sách ảnh' trong các file JSON nguồn
def build_image_map(json_sources):
    """Build {mã_tin -> [url, ...]} from raw crawler JSON files."""
    image_map = {}
    for src in json_sources:
        src = Path(src)
        if not src.exists():
            print(f'⚠️  Không tìm thấy file: {src}')
            continue
        rows = json.loads(src.read_text(encoding='utf-8'))
        for row in rows:
            ma_tin = str(row.get('Mã tin', '') or '').strip()
            if not ma_tin:
                continue
            raw = row.get('Danh sách ảnh', '[]') or '[]'
            try:
                urls = json.loads(raw) if isinstance(raw, str) else raw
            except Exception:
                urls = []
            if ma_tin in image_map:
                # merge, dedupe
                image_map[ma_tin] = list(dict.fromkeys(image_map[ma_tin] + urls))
            else:
                image_map[ma_tin] = urls
        print(f'✅ Loaded image map from {src} ({len(rows)} rows)')
    return image_map

def excel_to_csv_json(
    excel_path,
    csv_path=None,
    json_path=None,
    sheet_name=0,
    json_sources=None,
    id_column=None,
):
    """Convert Excel to CSV + JSON, drop '#' column, attach image_list from raw JSON sources."""
    excel_path = Path(excel_path)
    data = pd.read_excel(excel_path, sheet_name=sheet_name)

    # Bỏ cột '#' nếu tồn tại
    if '#' in data.columns:
        data = data.drop(columns=['#'])
        print('✅ Đã bỏ cột #')

    if csv_path is None:
        csv_path = excel_path.with_suffix('.csv')
    else:
        csv_path = Path(csv_path)
    if json_path is None:
        json_path = excel_path.with_suffix('.json')
    else:
        json_path = Path(json_path)

    # Tự nhận diện cột mã tin
    if id_column is None:
        for candidate in ['listing_id', 'Mã tin', 'Ma tin']:
            if candidate in data.columns:
                id_column = candidate
                break

    # Build image map từ JSON nguồn
    image_map = {}
    if json_sources:
        image_map = build_image_map(json_sources)

    def get_images(value):
        if pd.isna(value):
            return []
        key = str(value).strip()
        if re.fullmatch(r'\d+\.0', key):
            key = key.split('.')[0]
        return image_map.get(key, [])

    if id_column and id_column in data.columns:
        data['image_list'] = data[id_column].apply(get_images)
    else:
        data['image_list'] = [[] for _ in range(len(data))]

    data.to_csv(csv_path, index=False, encoding='utf-8-sig')
    data.to_json(json_path, orient='records', force_ascii=False, indent=2)

    mapped = (data['image_list'].apply(len) > 0).sum()
    total  = data['image_list'].apply(len).sum()
    print(f'✅ CSV  → {csv_path}')
    print(f'✅ JSON → {json_path}')
    print(f'✅ image_list: {mapped}/{len(data)} records, {total} ảnh tổng')
    return data, csv_path, json_path

In [6]:
# Save cleaned data
output_file = 'Final_Data.xlsx'
df_clean.to_excel(output_file, index=False)
print(f'✅ Cleaned data exported to: {output_file}  ({len(df_clean)} rows)')

# Export CSV + JSON: bỏ cột #, lấy image_list từ 2 file JSON nguồn
_, csv_output, json_output = excel_to_csv_json(
    output_file,
    csv_path='Final_Data.csv',
    json_path='Final_Data.json',
    json_sources=[
        'data/hcm_can_ho_full_detail_1.json',
        'data/hcm_nha_dat_full_detail_131.json',
    ],
)

✅ Cleaned data exported to: Final_Data.xlsx  (489 rows)
✅ Đã bỏ cột #
✅ Loaded image map from data/hcm_can_ho_full_detail_1.json (489 rows)
✅ Loaded image map from data/hcm_nha_dat_full_detail_131.json (2549 rows)
✅ CSV  → Final_Data.csv
✅ JSON → Final_Data.json
✅ image_list: 489/489 records, 4816 ảnh tổng
✅ Loaded image map from data/hcm_nha_dat_full_detail_131.json (2549 rows)
✅ CSV  → Final_Data.csv
✅ JSON → Final_Data.json
✅ image_list: 489/489 records, 4816 ảnh tổng
